In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import akita_utils_forplotting as utils
import numpy as np
import pandas as pd
import sys, os, psutil, time
import matplotlib.pyplot as plt
import pysam
    
from skimage.transform import resize


#from optparse import OptionParser


nt = ['A', 'T', 'C', 'G']

2024-07-25 11:46:42.207898: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-07-25 11:46:42.207924: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-07-25 11:46:46.253069: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2024-07-25 11:46:46.253095: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2024-07-25 11:46:46.253127: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (westway.gladstone.internal): /proc/driver/nvidia/version does not exist
2024-07-25 11:46:46.253297: I tensorflow/core/plat

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 sequence (InputLayer)          [(None, 1048576, 4)  0           []                               
                                ]                                                                 
                                                                                                  
 stochastic_reverse_complement   ((None, 1048576, 4)  0          ['sequence[0][0]']               
 (StochasticReverseComplement)  , ())                                                             
                                                                                                  
 stochastic_shift (StochasticSh  (None, 1048576, 4)  0           ['stochastic_reverse_complement[0
 ift)                                                            ][0]']                     

                                                                                                  
 re_lu_8 (ReLU)                 (None, 4096, 96)     0           ['max_pooling1d_7[0][0]']        
                                                                                                  
 conv1d_8 (Conv1D)              (None, 4096, 96)     46080       ['re_lu_8[0][0]']                
                                                                                                  
 batch_normalization_8 (BatchNo  (None, 4096, 96)    384         ['conv1d_8[0][0]']               
 rmalization)                                                                                     
                                                                                                  
 max_pooling1d_8 (MaxPooling1D)  (None, 2048, 96)    0           ['batch_normalization_8[0][0]']  
                                                                                                  
 re_lu_9 (

 add_2 (Add)                    (None, 512, 96)      0           ['add_1[0][0]',                  
                                                                  'dropout_2[0][0]']              
                                                                                                  
 re_lu_17 (ReLU)                (None, 512, 96)      0           ['add_2[0][0]']                  
                                                                                                  
 conv1d_17 (Conv1D)             (None, 512, 48)      13824       ['re_lu_17[0][0]']               
                                                                                                  
 batch_normalization_17 (BatchN  (None, 512, 48)     192         ['conv1d_17[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 re_lu_18 

 batch_normalization_25 (BatchN  (None, 512, 48)     192         ['conv1d_25[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 re_lu_26 (ReLU)                (None, 512, 48)      0           ['batch_normalization_25[0][0]'] 
                                                                                                  
 conv1d_26 (Conv1D)             (None, 512, 96)      4608        ['re_lu_26[0][0]']               
                                                                                                  
 batch_normalization_26 (BatchN  (None, 512, 96)     384         ['conv1d_26[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 dropout_7

                                                                                                  
 batch_normalization_32 (BatchN  (None, 512, 512, 48  192        ['conv2d_4[0][0]']               
 ormalization)                  )                                                                 
                                                                                                  
 dropout_9 (Dropout)            (None, 512, 512, 48  0           ['batch_normalization_32[0][0]'] 
                                )                                                                 
                                                                                                  
 add_9 (Add)                    (None, 512, 512, 48  0           ['symmetrize2d_1[0][0]',         
                                )                                 'dropout_9[0][0]']              
                                                                                                  
 symmetriz

 ormalization)                  )                                                                 
                                                                                                  
 dropout_12 (Dropout)           (None, 512, 512, 48  0           ['batch_normalization_38[0][0]'] 
                                )                                                                 
                                                                                                  
 add_12 (Add)                   (None, 512, 512, 48  0           ['symmetrize2d_4[0][0]',         
                                )                                 'dropout_12[0][0]']             
                                                                                                  
 symmetrize2d_5 (Symmetrize2D)  (None, 512, 512, 48  0           ['add_12[0][0]']                 
                                )                                                                 
          

In [22]:
import h5py
import pyBigWig
import intervaltree
import basenji
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# hic imports
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve

import collections 

In [4]:
import cooltools.lib.plotting
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import EngFormatter
import seaborn as sns


In [5]:
#REMEMBER!!!!!!! +1MB SEQUENCE GOES IN, BUT MAKES PREDICTION FOR 917504BP
#1MB CORRESPONDS TO 512, 2048 BP BINS THAT GETS CUT DOWN TO 448 BINS 

In [20]:
def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_bottom()
    if rotate:
        ax.tick_params(axis='x',rotation=45)
        
def save_figures_to_pdf(output_pdf_path, figures):
    with PdfPages(output_pdf_path) as pdf:
        for fig in figures:
            pdf.savefig(fig)

In [7]:
def read_fasta(fa):
    #read in fasta and return sequence only 
    seq=''
    with open(fa, "r") as fa_file:
        for line in fa_file:
            if not (line.startswith('>')):
                seq += line.strip()
    fa_file.close()                
    return(seq)
    

In [8]:
#TOTAL AKITA INPUT LENGTH 

akita_seq_len=1048576


#HG38 REF FA
hg38_file = '/pollard/home/shzhang/akita/run_akita/refs/hg38.fa'
fasta_hg38 = pysam.Fastafile(hg38_file)

#GENCODE
gene_annot = pd.read_csv('/pollard/data/genetics/GENCODE/release32/gencode.v32.genes_only.bed', sep = '\t', names = ['CHROM', 'start_gene', 'end_gene', 'Gene'])
gene_annot[gene_annot.CHROM != 'chrM']

,CHROM,start_gene,end_gene,Gene
0,chr1,11869,14409,DDX11L1
1,chr1,14404,29570,WASH7P
2,chr1,17369,17436,MIR6859-1
3,chr1,29554,31109,MIR1302-2HG
4,chr1,30366,30503,MIR1302-2
...,...,...,...,...
60567,chrY,57184101,57197337,IL9R
60568,chrY,57190738,57208756,AJ271736.1
60569,chrY,57201143,57203357,WASIR1
60570,chrY,57207346,57212230,WASH6P


In [9]:
data_dir='/pollard/data/projects/shzhang/mdanderson/'
patient_data_dir=f'{data_dir}by_patient_20240529/'

In [10]:
meta=pd.read_csv(f'{data_dir}meta_sv2.csv')
meta['tumor_id']=meta['tumor'].str.split('-').str.get(0)
meta

,Patient,Tumor_Type,Timepoint,normal,tumor,gender,tumor_id
0,Patient1,DDLPS,baseline,226294-WG01,226295-WG01,XX,226295
1,Patient1,DDLPS,On_treatment,226294-WG01,226296-WG01,XX,226296
2,Patient1,DDLPS,Surgery,226294-WG01,226298-WG01,XX,226298
3,Patient1,DDLPS,Surgery,226294-WG01,226297-WG01,XX,226297
4,Patient2,DDLPS,baseline,226299-WG01,226300-WG01,XX,226300
...,...,...,...,...,...,...,...
67,Patient23,DDLPS,baseline,227190-WG01,227180-WG01,XX,227180
68,Patient23,DDLPS,On_treatment,227190-WG01,227181-WG01,XX,227181
69,Patient23,DDLPS,Surgery,227190-WG01,227182-WG01,XX,227182
70,Patient24,UPS,baseline,227191-WG01,227183-WG01,XX,227183


In [11]:
test1=pd.read_csv(f'{patient_data_dir}walks_bed/{patient1_files[0][:-3]}.bed', sep='\t', 
                  header=None, names=['chr', 'start', 'end', 'strand', 'cn', 'idk'])
test1['span']=test1['end'] - test1['start']
test1

NameError: name 'patient1_files' is not defined

In [40]:

test1=pd.read_csv(f'{patient_data_dir}walks_bed/226309-WG01_walk_1_MYC_containing.bed', sep='\t', 
                  header=None, names=['chr', 'start', 'end', 'strand', 'cn', 'idk'])
test1['span']=test1['end'] - test1['start']
test1

,chr,start,end,strand,cn,idk,span
0,chr8,135877203,145138636,-,2,-166,9261433
1,chr8,133889203,135877202,-,3,-165,1987999
2,chr8,133260403,133889202,-,2,-164,628799
3,chr8,130575803,133260402,-,3,-163,2684599
4,chr8,129769803,130575802,-,2,-162,805999
5,chr8,128003603,129769802,-,3,-161,1766199
6,chr8,127560003,128003602,-,2,-160,443599
7,chr8,126801403,127560002,-,3,-159,758599
8,chr8,126268203,126801402,-,2,-158,533199
9,chr8,125657603,126268202,-,3,-157,610599


In [ ]:
for path in os.listdir(f'{patient_data_dir}patient_1/walks_fasta/'):
    print(path)


In [ ]:
#ok, what we want to do is plot the changes at every breakpoint
#we also want to figure out which bin the gene is in, so we can make a note of that
#we also want to get the bin at which the boundaries change, so we can also make a note of that 

In [31]:
#write separate plotting function so this isn't too messy
#for plotting post akita predictions 

def plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list):

    #order of lists is REF_left, REF_right, ALT
     
    fig=plt.figure(figsize=(12,6))
    target_index = 0
    vmin=-2; vmax=2

    plt.subplot(131) 
    im = plt.matshow(mat_list[2], fignum=False, cmap= 'RdBu_r', vmax=vmax, vmin=vmin)
    
    for g in gene_start_bin_list:
        plt.axvline(x = g, color = 'limegreen', linestyle = '--', linewidth=1, label = 'gene start')
    for g in gene_end_bin_list:
        plt.axvline(x = g, color = 'forestgreen', linestyle = '--', linewidth=1, label = 'gene end')
    for g in bkpt_bin_list:
        plt.axvline(x = g, color = 'blue', linestyle = '--', linewidth=1, label = 'other bkpts ')
    
    plt.colorbar(im, fraction=.04, pad = 0.05, ticks=[-2,-1, 0, 1,2]);
    plt.title(label_list[2], y=1.15)
    
    plt.subplot(132) 
    im = plt.matshow(mat_list[0], fignum=False, cmap= 'RdBu_r', vmax=vmax, vmin=vmin)
    plt.axvline(x = 224, color = 'blue', linestyle = '--', linewidth=1, label = 'gene start')

    plt.colorbar(im, fraction=.04, pad = 0.05, ticks=[-2,-1, 0, 1,2]);
    plt.title('REF Prediction Left: \n ' + label_list[0], y=1.15)
    

    plt.subplot(133) 
    im = plt.matshow(mat_list[1], fignum=False, cmap= 'RdBu_r', vmax=vmax, vmin=vmin)
    plt.axvline(x = 224, color = 'blue', linestyle = '--', linewidth=1, label = 'gene start')

    plt.colorbar(im, fraction=.04, pad = 0.05, ticks=[-2,-1, 0, 1,2]);
    plt.title('REF Prediction Right' + '\n' + label_list[1], y=1.15 )
    
    
    fig.suptitle(label_list[3], fontsize=16, y=0.90)

        
    plt.tight_layout()
    plt.subplots_adjust(top=0.75)
    return(fig)

In [38]:
#make akita predictions for each walk and plot 
def pred_akita_bkpts(fa_dir, fa_name, patient_num):
    bed_name=fa_name.replace('fa', 'bed')
    name_parts = bed_name.rsplit('_', 2)

    gencode_coord=gene_annot[gene_annot.Gene == name_parts[1]]    
    gene_len=abs(gencode_coord.end_gene-gencode_coord.start_gene)
    gene_mid_coord=np.floor(gencode_coord.start_gene + (gene_len/2))
    #print(gencode_coord)
    
    try:

        #read in bed file and figure out where to center the file 
        bed=pd.read_csv(f'{patient_data_dir}walks_bed/' + bed_name, sep='\t', header=None)
        bed.columns = ['chr', 'start', 'end', 'strand', 'cn', '5']
        bed['chr'] = bed['chr'].astype('str') 
        bed['span']=abs(bed['end']-bed['start']) 
        bed['span_cumsum'] = bed['span'].cumsum()
    
        
        #print(bed)
        
        
        #get the row where gene_mid_coord 
        start_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.start_gene)) & (int(gencode_coord.start_gene) <= bed['end']) ) 
        row_gene_start_list = start_condition.index[start_condition].tolist()
            
        end_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.end_gene)) & (int(gencode_coord.end_gene) <= bed['end']) ) 
        row_gene_end_list = end_condition.index[end_condition].tolist()
        #print('row_gene_start_list', row_gene_start_list)
    
    except: 
        print('bed file not found!!!!!!!!')
       

    
    # read in fasta file 
    seq=read_fasta(f'{fa_dir}{fa_name}')
            
    
    figures=[]
    for i in range (0, len(bed)-1):
    #for i in range(0,4):
        #print('row ', i)
        
        try:
        
            #get first, last rows, and subsets of bed before and after bkpt 
            first_row=bed.iloc[0,:]
            last_row=bed.iloc[(len(bed)-1), :]
            left_bkpt=bed.iloc[i, :]
            right_bkpt=bed.iloc[i+1,:]
            bed_to_bkpt=bed.iloc[0:(i+1),:]
        
            #get relevant lengths
            total_span=np.sum(bed['span'])
            len_left_to_bkpt=np.sum(bed_to_bkpt['span'])
            len_bkpt_to_right=total_span-len_left_to_bkpt
        
        
        
            # left sequence 
            if len_left_to_bkpt>=utils.half_patch_size:
                ALT_left=seq[(len_left_to_bkpt - utils.half_patch_size ):len_left_to_bkpt]
            
            else:   
                #need to pad left from reference 
                diff_len=utils.half_patch_size-len_left_to_bkpt
           
                first_row_idx=first_row.start
                ALT_fromref_left=fasta_hg38.fetch(first_row.chr, first_row_idx-diff_len, first_row_idx).upper()
            
                ALT_left=ALT_fromref_left + seq[0:len_left_to_bkpt]
            
        
            # right sequence 
            if len_bkpt_to_right>utils.half_patch_size:
                ALT_right=seq[len_left_to_bkpt: (len_left_to_bkpt + utils.half_patch_size)]
            
            else:
                #need to pad right from reference        
                diff_len=utils.half_patch_size-len_bkpt_to_right
            
                last_row_idx=last_row.end
                ALT_fromref_right=fasta_hg38.fetch(last_row.chr, last_row_idx, last_row_idx + diff_len).upper()

                ALT_right=seq[len_left_to_bkpt:total_span] + ALT_fromref_right
            
            
            ALT_seq = ALT_left + ALT_right

            
            ########################

            #we want to get indices of the breakpoints, if others are in the window 

            #get index for where the akita window starts 
            left_start=len_left_to_bkpt-utils.half_patch_size
            #print(left_start)

            bkpt_pos_from_left_all=bed['span_cumsum'].tolist() - left_start 
            #print(bkpt_pos_from_left_all)

            #subset to those less than akita window size
            bkpt_pos_from_left=bkpt_pos_from_left_all[(bkpt_pos_from_left_all>0) & (bkpt_pos_from_left_all<utils.MB)]
            #print(bkpt_pos_from_left)
            
            #get the bin index
            bkpt_bin_list_all=[utils.get_bin(bkpt) for bkpt in bkpt_pos_from_left]
            bkpt_bin_list = [bkpt for bkpt in bkpt_bin_list_all if 0 < bkpt <= 448]


            
            ########################
            #finally, get the positions of the gene(s)
                       
            gene_start_bin_list=[]
            gene_end_bin_list=[]

            try:
                
                #check the start of the gene 
                for g in row_gene_start_list:
                    len_to_gene_start=gencode_coord.start_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene start' , len_to_gene_start)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_start+=len_before_row


                    gene_start_pos_window=len_to_gene_start - left_start 
                    if (gene_start_pos_window>0) & (gene_start_pos_window<utils.MB):
                        gene_start_bin_list.append(utils.get_bin(gene_start_pos_window))
                        
                #check the end of the gene      
                for g in row_gene_end_list:
                    len_to_gene_end=gencode_coord.end_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene end' , len_to_gene_end)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_end+=len_before_row

                    gene_end_pos_window=len_to_gene_end - left_start 
                    if (gene_end_pos_window>0) & (gene_end_pos_window<utils.MB):
                        gene_end_bin_list.append(utils.get_bin(gene_end_pos_window))

                
            except:
                print("error in gene annotation!")


            #########################
            #finally, get the REF -- for the two sequences to the left and right of the breakpoint 
            left_ref_coords=[left_bkpt.chr, left_bkpt.end-utils.half_patch_size, 
                                          left_bkpt.end+utils.half_patch_size]
            right_ref_coords=[right_bkpt.chr, right_bkpt.start-utils.half_patch_size, 
                                          right_bkpt.start+utils.half_patch_size]


            REF_seq_left=fasta_hg38.fetch(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]).upper()
            REF_seq_right=fasta_hg38.fetch(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]).upper()


            ####################

            #check sequence lengths, and then run akita 
            #print(len(REF_seq_left))
            assert len(REF_seq_left) == utils.seq_length, 'REF_seq_left is not right length'
            assert len(REF_seq_right) == utils.seq_length, 'REF_seq_right is not right length'
            assert len(ALT_seq) == utils.seq_length, 'ALT_seq is not right length'



            #this is where akita is actually run, output vector size is (99681,)
            REF_left_vec, REF_right_vec, ALT_vec = utils.vector_from_seq(REF_seq_left), utils.vector_from_seq(REF_seq_right), utils.vector_from_seq(ALT_seq)
            REF_left_mat, REF_right_mat, ALT_mat = utils.mat_from_vector(REF_left_vec), utils.mat_from_vector(REF_right_vec), utils.mat_from_vector(ALT_vec)

    
            #order is REF_left, REF_right, ALT
            mat_list=[REF_left_mat, REF_right_mat, ALT_mat]
            
            #add annotation for span between bkpts if same chr, or annot as translocation
            if left_ref_coords[0]==left_ref_coords[0]:
                span_annot=str(right_ref_coords[1] - left_ref_coords[2]) + ' bp between breakpoints'
            else:
                span_annot='interchromosomal'
            
            #get label list 
            label_list=['{}:{}-{}'.format(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]), 
                       '{}:{}-{}'.format(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]), 
                       'Left: {}:{}-{} \n Right: {}:{}-{} \n {}'.format(
                        left_bkpt.chr, left_bkpt.start, left_bkpt.end, 
                           right_bkpt.chr, right_bkpt.start, right_bkpt.end, 
                           span_annot), f'{fa_name[:-3]} \n rows {i} - {i+1}']

            #print(label_list)
            # plot!
            fig=plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list)
            figures.append(fig)
           

        
        except:
            print('error!!!! in row index', i )
            
    #print(f'saving to {fig_dir}patient_{patient_num}/{bed_name[:-4]}_breakpoints.pdf')
    save_figures_to_pdf(f'{fig_dir}patient_{patient_num}/{bed_name[:-4]}_plot.pdf', figures)
    plt.close('all')

In [39]:
fig_dir='/pollard/data/projects/shzhang/mdanderson/figs/by_patient_20240529/'

#run for all patients
for patient_num in range(1,25):
    print('running patient ', patient_num)
    dir_patient=f'{patient_data_dir}patient_{patient_num}/walks_fasta/'
    dir_files=os.listdir(dir_patient)
    
    #run for all walks in patient 
    for f in range (0, len(dir_files)):
        print('running file ', dir_files[f])
        pred_akita_bkpts(fa_dir=dir_patient, fa_name=dir_files[f], patient_num=patient_num)

running patient  1
running file  226296-WG01_walk_5_MYCL_containing.fa
running file  226298-WG01_walk_1_MYC_containing.fa
running file  226297-WG01_walk_1_KRAS_containing.fa
running file  226298-WG01_walk_1_MYCL_containing.fa


/tmp/ipykernel_1420699/4090048388.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(12,6))


error!!!! in row index 42
running file  226296-WG01_walk_4_MYCL_containing.fa
running file  226296-WG01_walk_1_MYCN_containing.fa
error!!!! in row index 0
error!!!! in row index 2
running file  226296-WG01_walk_2_MYCN_containing.fa
running file  226297-WG01_walk_1_MDM2_containing.fa
running file  226296-WG01_walk_1_MYCL_containing.fa
running file  226295-WG01_walk_1_MYCL_containing.fa
error!!!! in row index 8
running file  226298-WG01_walk_1_MYCN_containing.fa
running file  226295-WG01_walk_1_MYCN_containing.fa
running file  226298-WG01_walk_2_MYCL_containing.fa
running file  226297-WG01_walk_1_MYCN_containing.fa
running file  226296-WG01_walk_3_MYCL_containing.fa
error!!!! in row index 5
running file  226296-WG01_walk_2_MYCL_containing.fa
error!!!! in row index 8
running file  226295-WG01_walk_1_MYC_containing.fa
running file  226297-WG01_walk_1_MYC_containing.fa
running file  226297-WG01_walk_1_MYCL_containing.fa
running file  226296-WG01_walk_2_MYC_containing.fa
running file  226295

/tmp/ipykernel_1420699/4090048388.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(12,6))


error!!!! in row index 31
running file  226300-WG01_walk_3_MYCL_containing.fa
error!!!! in row index 14
running file  226301-WG01_walk_2_MYCN_containing.fa
error!!!! in row index 0
running file  226301-WG01_walk_3_MYC_containing.fa
error!!!! in row index 0
running file  226301-WG01_walk_2_MDM2_containing.fa


...

/tmp/ipykernel_1420699/4090048388.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(12,6))


error!!!! in row index 23
error!!!! in row index 24
running file  227184-WG01_walk_3_MYC_containing.fa
running file  227183-WG01_walk_1_MYCL_containing.fa
running file  227183-WG01_walk_2_MYC_containing.fa
running file  227183-WG01_walk_1_MYCN_containing.fa
error!!!! in row index 0
error!!!! in row index 2
running file  227184-WG01_walk_2_MYCL_containing.fa


/tmp/ipykernel_1420699/4090048388.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(12,6))


running file  227183-WG01_walk_2_MYCL_containing.fa
running file  227184-WG01_walk_2_MYC_containing.fa


/tmp/ipykernel_1420699/4090048388.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(12,6))


In [111]:
patient1_files=os.listdir(f'{patient_data_dir}/patient_1/walks_fasta')
patient1_files